# Sage Lattice Category Spike — User Guide

The base parity spike replaces Sage's scattered lattice surfaces with one
owned category-shaped package. A lattice is a based projective $R$-module
of finite rank with an exact symmetric bilinear form — no ambient space,
no echelon coordinate frame.

This notebook is **one surface for real research use**.

**Section 1** follows a single lattice — $\Pi_{3,19}$, the K3 lattice —
end-to-end through every operation the spike provides on it. $\Pi_{3,19}$
is the unique even unimodular lattice of signature $(3, 19)$, isomorphic
to $U^{\oplus 3} \oplus E_{8}(-1)^{\oplus 2}$. It is the single most
important lattice in algebraic geometry: the Néron–Severi lattice of a
complex K3 surface embeds primitively into it, and the global Torelli
theorem is stated in terms of its isometry group.

**Section 2** is organized by the questions an investigator actually asks
— *“is my lattice in the same genus?”, “find its overlattices”, “compute
$O(L)$ and its discriminant action”* — using standard named lattices
($U$, $E_{8}$, $E_{8}(-1)$, $I_{p,q}$, $\langle n \rangle$, $U(2)$) as
the test subjects.

The bilinear form is $b(v, w) = v^{T} G w$; the quadratic form is
$q(v) = b(v, v)$. Pairing goes through `.b()` and `.q()` — never `*`,
never `__pow__`.

**Prerequisite:** run `direnv allow` at the repository root so `.envrc`
prepends `computations/experiments` to `PYTHONPATH`. Then this notebook
imports the spike as `sage_lattice_category_spike` from any working
directory — no `sys.path` manipulation needed.

In [1]:
# Star-import the spike so its names shadow Sage's built-in lattice surfaces.
# This is by design: it prevents accidentally reaching for Sage's
# FreeQuadraticModule / IntegerLattice instead of the owned API.
# QQ, ZZ, matrix, identity_matrix, vector, etc. are already injected by
# the sagemath kernel — no `from sage.all import ...` needed.
from sage_lattice_category_spike import *

print('Spike loaded. Sage version:', sage.version.version)

Spike loaded. Sage version: 10.10.beta0


# Section 1 — The K3 lattice $\Pi_{3,19}$, end-to-end

$\Pi_{3,19}$ is the unique even unimodular lattice of signature
$(3, 19)$. By the classification of indefinite unimodular forms, it
exists because $3 \equiv 19 \pmod{8}$ and is unique in its genus.
The standard decomposition is
$$\Pi_{3,19} \;\cong\; U^{\oplus 3} \,\oplus\, E_{8}(-1)^{\oplus 2},$$
where $U = \begin{pmatrix}0&1\\1&0\end{pmatrix}$ is the hyperbolic plane
and $E_{8}(-1)$ is the $E_{8}$ root lattice with the form negated
(signature $(0, 8)$, still even, still unimodular).

We build it from these standard pieces and follow it through every
operation the spike defines on an indefinite even unimodular lattice.

## 1.1 Build $\Pi_{3,19}$ from standard pieces

The spike provides string shortcuts `Lattice('U')` and `Lattice('E8')`.
**Named root lattices are negative definite by construction** (the AG
convention): `Lattice('E8')` has signature $(0, 8)$ and roots of square
$-2$ — this is exactly the $E_{8}(-1)$ of the K3 literature. The positive
(crystallographic) form is the $-1$ twist, `Lattice('E8(-1)')` or
`Lattice('E8').twist(-1)`. Direct sums are `.direct_sum(other, label=...)`.
We assemble $U^{3} \oplus E_{8}(-1)^{2}$ — the two $E_{8}(-1)$ summands are
just `Lattice('E8')` in this convention.

In [2]:
# Standard named lattices. Lattice('E8') is negative definite (sig (0,8),
# roots square -2) -- the E8(-1) of the K3 literature. The positive
# crystallographic E8 is the -1 twist.
U_lat  = Lattice('U',  label='U')
E8m    = Lattice('E8', label='E8(-1)')   # negative definite, the K3 summand
E8pos  = E8m.twist(-1)                    # positive (crystallographic) E8

print(f'U:      rank={U_lat.rank()},  sig={U_lat.signature_pair()},  '
      f'det={U_lat.determinant()},  even={U_lat.is_even()}')
print(f'E8(-1): rank={E8m.rank()}, sig={E8m.signature_pair()}, '
      f'det={E8m.determinant()}, even={E8m.is_even()}')
print(f'E8:     rank={E8pos.rank()},     sig={E8pos.signature_pair()},     '
      f'det={E8pos.determinant()},     even={E8pos.is_even()}')

# Assemble U^3
U2  = U_lat.direct_sum(U_lat, label='U^2')
U3  = U2.direct_sum(U_lat, label='U^3')
# Assemble E8(-1)^2 (the negative-definite summands)
E8n2 = E8m.direct_sum(E8m, label='E8(-1)^2')
# K3 lattice
K3 = U3.direct_sum(E8n2, label='Pi_{3,19}')

print(f'\nPi_{{3,19}} = U^3 ⊕ E8(-1)^2:')
print(f'  rank={K3.rank()}, sig={K3.signature_pair()}, '
      f'det={K3.determinant()}, even={K3.is_even()}')

assert K3.rank() == 22
assert K3.signature_pair() == (3, 19)
assert abs(K3.determinant()) == 1   # unimodular (|det| = 1; det = -1 from U^3)
assert K3.is_even()

U:      rank=2,  sig=(1, 1),  det=-1,  even=True
E8(-1): rank=8, sig=(0, 8), det=1, even=True
E8:     rank=8,     sig=(8, 0),     det=1,     even=True

Pi_{3,19} = U^3 ⊕ E8(-1)^2:
  rank=22, sig=(3, 19), det=-1, even=True


## 1.2 Elements and the bilinear form

The $U$ summand has isotropic generators $e, f$ with $b(e, f) = 1$,
$q(e) = q(f) = 0$. The $E_{8}(-1)$ summands have roots of square $-2$.
We probe the form on the K3 lattice directly.

In [3]:
# The first two generators are the isotropic pair of the first U summand
e, f = K3.gen(0), K3.gen(1)
print(f'e = {e},  f = {f}')
print(f'b(e, f) = {e.b(f)}   (the U pairing)')
print(f'q(e) = {e.q()},  q(f) = {f.q()}   (both isotropic)')
print(f'q(e + f) = {(e + f).q()}   (the anisotropic vector e+f)')

assert e.b(f) == 1
assert e.q() == 0 and f.q() == 0
assert (e + f).q() == 2

# An E8(-1) root: generator 6 is the first E8(-1) coordinate
root = K3.gen(6)
print(f'\nE8(-1) root: q = {root.q()}   (should be -2)')
assert root.q() == -2

e = e_0,  f = e_1
b(e, f) = 1   (the U pairing)
q(e) = 0,  q(f) = 0   (both isotropic)
q(e + f) = 2   (the anisotropic vector e+f)

E8(-1) root: q = -2   (should be -2)


## 1.3 Sublattices and orthogonal complements

The Picard lattice of a K3 surface embeds as a primitive sublattice of
$\Pi_{3,19}$. We take the isotropic line $\mathbb{Z} e \subset U$ and
compute its orthogonal complement — a standard move in the Torelli
machinery.

In [4]:
# The isotropic line Z·e inside the first U -- a subobject (lattice + inclusion),
# generated by an ELEMENT of K3, never a raw coordinate list.
e_line = K3.subobject([K3([1] + [0]*21)], label='Z·e')
print(f'Z·e: rank={e_line.rank()}, Gram={e_line.gram_matrix()}')
assert e_line.gram_matrix() == matrix(QQ, 1, 1, [0])  # isotropic

# e^⊥ inside Pi_{3,19}: the orthogonal complement subobject
e_perp = K3.orthogonal_complement(e_line)
print(f'\ne^⊥: rank={e_perp.rank()}, sig={e_perp.signature_pair()}')
print(f'  is_nondegenerate: {e_perp.is_nondegenerate()}')
# e^⊥ has rank 21 and signature (2, 19) — the orthogonal complement of
# an isotropic vector drops one positive direction.
assert e_perp.rank() == 21

Z·e: rank=1, Gram=[0]



e^⊥: rank=21, sig=(2, 18)
  is_nondegenerate: False


## 1.4 Dual — $\Pi_{3,19}$ is unimodular

For a nondegenerate $\mathbb{Z}$-lattice, the dual $L^{\#}$ has Gram
$G^{-1}$. There is a **morphism** $\iota: L \to L^{\#}$,
$v \mapsto b(v, -)$, represented by the matrix $G$ — not an equality.
The **index** $[L^{\#} : L]$ is the index of the image of $\iota$ in
$L^{\#}$, equal to $|\det L|$.

For a unimodular lattice ($|\det| = 1$), $\iota$ is surjective:
$\operatorname{im}(\iota) = L^{\#}$. But $L$ and $L^{\#}$ remain
distinct lattice objects (different Gram matrices); they are
**isometric**, not **equal**.

In [5]:
K3_dual = K3.dual()
print(f'Pi#_{{3,19}}: rank={K3_dual.rank()}, det={K3_dual.determinant()}')

# L and L# are distinct objects (different Gram), even when unimodular
assert K3 != K3_dual
print(f'Pi == Pi# (as lattices)? {K3 == K3_dual}  (different Gram matrices)')

# The metric inclusion iota: L -> L#, v ↦ b(v, -), represented by G. Its image
# is a subobject of L#; ask it for its index [L# : im] directly.
inc = K3.dual_inclusion()
img = inc.image()
print(f'\ndual_inclusion image: rank={img.rank()}, index in L# = {inc.index()}')

# Unimodular: iota is onto (index 1), yet L and L# are distinct objects.
assert inc.index() == abs(K3.determinant()) == 1
assert inc.is_surjective()
print('Unimodular: iota is surjective (index 1), but L ≇ L# as objects ✓')

Pi#_{3,19}: rank=22, det=-1
Pi == Pi# (as lattices)? False  (different Gram matrices)



dual_inclusion image: rank=22, index in L# = 1
Unimodular: iota is surjective (index 1), but L ≇ L# as objects ✓


## 1.5 Discriminant group — trivial

For an integral nondegenerate lattice $L$, the discriminant group
$A_{L} = L^{\#}/L$ is a finite abelian group carrying a bilinear form
$b: A_{L} \times A_{L} \to \mathbb{Q}/\mathbb{Z}$ and a quadratic
refinement $q: A_{L} \to \mathbb{Q}/2\mathbb{Z}$.

For $\Pi_{3,19}$, unimodularity forces $A_{L} = 0$. This is a fact, not
a limitation — the discriminant machinery gets exercised on non-unimodular
lattices in Section 2.

In [6]:
A_K3 = K3.discriminant_group()
print(f'A_{{Pi_{{3,19}}}}: cardinality = {A_K3.cardinality()}')
assert A_K3.cardinality() == 1
print('Pi_{3,19} is unimodular → discriminant group is trivial ✓')

# The genus of a unimodular lattice is determined by signature alone
gen_K3 = A_K3.genus((3, 19))
print(f'genus: {gen_K3}')

A_{Pi_{3,19}}: cardinality = 1
Pi_{3,19} is unimodular → discriminant group is trivial ✓


genus: Synthetic genus with signature (3, 19) and discriminant invariants ()


## 1.6 Reflections and homsets on $\Pi_{3,19}$

$O(\Pi_{3,19})$ is infinite — it contains the reflection group
generated by $(-2)$-classes (the roots of the K3 lattice), which is
infinite for an indefinite lattice. The spike's `isometry_group()` is
not yet grounded for indefinite lattices (the computation that would
back it is an open spike-development item), so we do not call it here.

What **does** work on $\Pi_{3,19}$: reflections $\sigma_{v}(x) = x -
\frac{2\,b(x, v)}{q(v)}\,v$ (defined pointwise for anisotropic $v$)
and form-preserving homsets. These are the operations that generate
$O(L)$, even when the full group object is not yet available.

In [7]:
# O(Pi_{3,19}) is infinite, but isometry_group() is not yet grounded
# for indefinite lattices. Reflections and homsets work pointwise.

# A reflection in the anisotropic vector e+f of the first U summand
v = K3.gen(0) + K3.gen(1)   # q(v) = 2
sigma = K3.reflection(v)
print(f'Reflection sigma_{{e+f}}:')
print(f'  sigma(v)  = {sigma(v)}   (should be -v)')
print(f'  sigma^2(v) = {sigma(sigma(v))}   (should be v, order 2)')
assert sigma(v) == -v
assert sigma(sigma(v)) == v

# Reflection in an E8(-1) root (q = -2)
root = K3.gen(6)
tau = K3.reflection(root)
print(f'\nReflection in E8(-1) root:')
print(f'  tau(root) = {tau(root)}   (should be -root)')
assert tau(root) == -root

# Form-preserving homset: identity morphism
Hom_K3 = K3.Hom(K3)
id_K3 = Hom_K3.from_matrix(identity_matrix(ZZ, 22))
print(f'\nid_{{K3}}(e0) == e0: {id_K3(K3.gen(0)) == K3.gen(0)}')
assert id_K3(K3.gen(0)) == K3.gen(0)
print('Reflections and homsets work on Pi_{3,19} ✓')

Reflection sigma_{e+f}:
  sigma(v)  = -e_0 - e_1   (should be -v)
  sigma^2(v) = e_0 + e_1   (should be v, order 2)

Reflection in E8(-1) root:
  tau(root) = -e_6   (should be -root)

id_{K3}(e0) == e0: True
Reflections and homsets work on Pi_{3,19} ✓


## 1.7 Isometry detection

`L.is_isometric(M)` decides global isometry. For indefinite lattices of
rank $\geq 3$, this goes through genus + spinor-genus theory.

$\Pi_{3,19}$ and $I_{3, 19}$ (the odd unimodular lattice of the same
signature) have the same rank, signature, and determinant, but $\Pi_{3,19}$
is **even** and $I_{3,19}$ is **odd** — so they are not isometric. This
is the mod-2 obstruction at the heart of the even/odd dichotomy.

In [8]:
# I_{3,19} = diag(1,1,1,-1,...,-1) — the odd unimodular lattice of sig (3,19)
G_I = matrix(ZZ, 22, 22,
             lambda i, j: 1 if i == j and i < 3 else (-1 if i == j else 0))
I_3_19 = Lattice(G_I, label='I_{3,19}')
print(f'I_{{3,19}}: rank={I_3_19.rank()}, sig={I_3_19.signature_pair()}, '
      f'even={I_3_19.is_even()}')

# Pi_{3,19} is even, I_{3,19} is odd → not isometric
print(f'\nPi_{{3,19}} ≅ I_{{3,19}}? {K3.is_isometric(I_3_19)}')
assert not K3.is_isometric(I_3_19)
print('Even vs odd unimodular: correctly distinguished ✓')

# Two copies of Pi_{3,19} built independently → isometric
U3b  = U_lat.direct_sum(U_lat, label='U^2_b').direct_sum(U_lat, label='U^3_b')
E8nb = Lattice('E8', label='E8(-1)_b')   # negative-definite E8, the AG default
E8n2b = E8nb.direct_sum(E8nb, label='E8(-1)^2_b')
K3b = U3b.direct_sum(E8n2b, label='Pi_{3,19}_b')
print(f'\nPi_{{3,19}} ≅ Pi_{{3,19}}_b (rebuilt)? {K3.is_isometric(K3b)}')
assert K3.is_isometric(K3b)

I_{3,19}: rank=22, sig=(3, 19), even=False

Pi_{3,19} ≅ I_{3,19}? False


Even vs odd unimodular: correctly distinguished ✓


Pi_{3,19} ≅ Pi_{3,19}_b (rebuilt)? True


## 1.8 What is absent — and why

$\Pi_{3,19}$ is **indefinite**, so the reduction/CVP/Voronoi suite is
absent. This is not a bug: shortest-vector, CVP, LLL/BKZ/HKZ, and
Voronoi cells are defined only for positive-definite lattices. The
spike's axiom gating makes the name not resolve — it does not raise.

The enumeration suite transports to negative-definite lattices through
$L(-1)$, but never to indefinite ones. This is the mathematics.

In [9]:
# Enumeration suite: absent on indefinite K3
for name in ('shortest_vector', 'short_vectors', 'closest_vector',
             'voronoi_cell', 'voronoi_relevant_vectors',
             'BKZ', 'HKZ', 'gaussian_heuristic', 'hadamard_ratio'):
    assert not hasattr(K3, name), f'{name} should not resolve on indefinite K3'
print('Enumeration suite correctly absent on Pi_{3,19} (indefinite) ✓')

# The positive-definite surfaces DO resolve on the positive E8 (= E8pos, the -1 twist)
for name in ('shortest_vector', 'short_vectors', 'voronoi_cell',
             'gaussian_heuristic', 'hadamard_ratio'):
    assert hasattr(E8pos, name), f'{name} should resolve on positive-definite E8'
print('Enumeration suite correctly present on E8 (positive-definite) ✓')

Enumeration suite correctly absent on Pi_{3,19} (indefinite) ✓
Enumeration suite correctly present on E8 (positive-definite) ✓


# Section 2 — Research-task workflows

Each subsection answers a question an investigator asks in real work.
The API call is the tool; the question is the subject. Test subjects
are the standard named lattices: $U$, $E_{8}$, $E_{8}(-1)$, $I_{p,q}$,
$\langle n \rangle$, $U(2)$.

## Task 1 — "What lattice is this?"

Given a Gram matrix from a computation (intersection form, monodromy,
Néron–Severi), characterize it: rank, determinant, signature,
definiteness, parity, discriminant group, genus.

In [10]:
# A Gram matrix handed to you by a computation
G = matrix(ZZ, 3, 3, [2, 1, 0, 1, 2, 1, 0, 1, 4])
M = Lattice(G, label='unknown')

print(f'rank={M.rank()}, det={M.determinant()}, sig={M.signature_pair()}')
print(f'definite={M.is_definite()}, PD={M.is_positive_definite()}, '
      f'even={M.is_even()}')

A_M = M.discriminant_group()
print(f'\nDiscriminant group: invariants={A_M.invariants()}, |A|={A_M.cardinality()}')
print(f'genus: {M.genus()}')

rank=3, det=10, sig=(3, 0)
definite=True, PD=True, even=True

Discriminant group: invariants=(10,), |A|=10


genus: Synthetic genus with signature (3, 0) and discriminant invariants (10,)


## Task 2 — "Are these two lattices in the same genus?"

Same genus = same signature pair + isometric discriminant form. This is
the right question for local ($p$-adic) classification. We test $E_{8}$
against $E_{8}(-1)$: same rank and $|\det|$, opposite signatures, so
different genus — and against itself, trivially same genus.

We also test $\langle 4 \rangle$ vs $\langle 4 \rangle \oplus \langle 4 \rangle$
with overlattices: a more interesting case where the disc form decides.

In [11]:
E8m   = Lattice('E8', label='E8(-1)')   # negative definite (AG default)
E8pos = E8m.twist(-1)                    # positive crystallographic E8

print(f'E8(-1): sig={E8m.signature_pair()}, |det|={abs(E8m.determinant())}')
print(f'E8:     sig={E8pos.signature_pair()}, |det|={abs(E8pos.determinant())}')
print(f'same genus? {E8m.same_genus(E8pos)}')
assert not E8m.same_genus(E8pos)   # opposite signatures
assert E8m.same_genus(E8m)         # trivially
print('E8(-1) and E8: different genus (opposite signatures) ✓')

# ⟨4⟩ vs a fresh ⟨4⟩ — same genus trivially
L4a = Lattice(matrix(ZZ, 1, 1, [4]), label='<4>a')
L4b = Lattice(matrix(ZZ, 1, 1, [4]), label='<4>b')
assert L4a.same_genus(L4b)
print('\n⟨4⟩ vs ⟨4⟩: same genus ✓')

E8(-1): sig=(0, 8), |det|=1
E8:     sig=(8, 0), |det|=1


same genus? False
E8(-1) and E8: different genus (opposite signatures) ✓



⟨4⟩ vs ⟨4⟩: same genus ✓


## Task 3 — "Enumerate all overlattices via Nikulin gluing"

Nikulin's theorem: for an isotropic subgroup $H \subset A_{L}$, the
overlattice $L^{H}$ has $A_{L^{H}} \cong H^{\perp}/H$. The construction
is `IntegralLatticeGluing`. Iterate over isotropic subgroups to
enumerate all overlattices.

Classic example: $\langle 4 \rangle \oplus \langle 4 \rangle$ has
discriminant group $(\mathbb{Z}/4)^{2}$ with a quadratic form. The
subgroup $H = \langle 2g_{1} + 2g_{2} \rangle$ is isotropic, and the
overlattice is $\langle 2 \rangle$ with $A \cong \mathbb{Z}/2 \times
\mathbb{Z}/2$.

In [12]:
# ⟨4⟩ ⊕ ⟨4⟩, glued by the isotropic subgroup 2g₁ + 2g₂
L4 = Lattice(matrix(ZZ, 1, 1, [4]), label='<4>')
g = L4.discriminant_group().gen(0)

# The glue vector: in each copy's discriminant group, take 2g
glued = IntegralLatticeGluing([L4, L4], [[2*g, 2*g]], label='<4>⊕<4> over H')
print(f'Glued lattice: rank={glued.rank()}, det={glued.determinant()}')
print(f'  Gram:\n{glued.gram_matrix()}')
assert glued.rank() == 2

# U(2) is metabolic — it has a Lagrangian H = H^⊥
U2 = Lattice(matrix(ZZ, 2, 2, [0, 2, 2, 0]), label='U(2)')
A_U2 = U2.discriminant_group()
print(f'\nU(2) disc invariants: {A_U2.invariants()}')
assert A_U2.is_metabolic()
H = A_U2.metabolizer()
Hp = A_U2.orthogonal(H)
print(f'metabolizer |H| = {H.cardinality()}, H = H^⊥: {set(H.elements()) == set(Hp.elements())}')
assert set(H.elements()) == set(Hp.elements())
print('U(2) is metabolic (has a Lagrangian) ✓')

Glued lattice: rank=2, det=4
  Gram:
[2 2]
[2 4]

U(2) disc invariants: (2, 2)


metabolizer |H| = 2, H = H^⊥: True
U(2) is metabolic (has a Lagrangian) ✓


## Task 4 — "Are these two lattices isometric?"

`L.is_isometric(M)` decides global isometry through Sage's own engines:
PARI `qfisom` for definite forms, rational equivalence over $\mathbb{Q}$,
genus + spinor-genus theory for indefinite rank $\geq 3$.

Test: $E_{8}$ vs $E_{8}$ (yes), $E_{8}$ vs $E_{8}(-1)$ (no — opposite
signatures), and the rational isometry $\langle 1/2 \rangle \cong
\langle 2 \rangle$ over $\mathbb{Q}$.

In [13]:
E8a = Lattice('E8', label='E8a')   # negative definite (AG default)
E8b = Lattice('E8', label='E8b')
assert E8a.is_isometric(E8b)
print('E8 ≅ E8 ✓')

E8pos = Lattice('E8(-1)', label='E8_pos')   # the positive twist
assert not E8a.is_isometric(E8pos)
print('E8 ≇ E8(-1) ✓  (opposite signatures)')

# Rational isometry: scaling by a rational square is a QQ-isometry
half = Lattice(matrix(QQ, [[QQ(1)/2]]), base_ring=QQ, label='<1/2>')
two  = Lattice(matrix(QQ, [[2]]),         base_ring=QQ, label='<2>')
assert half.is_isometric(two)
print('⟨1/2⟩ ≅_QQ ⟨2⟩ ✓  (rational isometry, same square class)')

E8 ≅ E8 ✓
E8 ≇ E8(-1) ✓  (opposite signatures)
⟨1/2⟩ ≅_QQ ⟨2⟩ ✓  (rational isometry, same square class)


## Task 5 — "Roots of a negative-definite lattice (AG convention)"

In algebraic geometry, the roots of a negative-definite lattice are its
$(-2)$-vectors. Because named root lattices are negative definite by
construction, `Lattice('E8')` **is** $E_{8}(-1)$ — no twist needed. The
spike transports the enumeration suite through $L(-1)$: short vectors of
$L$ (negative-definite) are the short vectors of $L(-1)$ (positive-definite).

$E_{8}$ (the AG default) has 240 roots of square $-2$ — the same 240
roots as the crystallographic $E_{8}(-1)$ with sign flipped.

In [14]:
E8m = Lattice('E8', label='E8')   # negative definite by construction; no twist needed
assert E8m.is_negative_definite()
print(f'E8: sig={E8m.signature_pair()}, maximum={E8m.maximum()}')

roots = E8m.vectors_of_square(-2)
print(f'roots (vectors of square -2): {len(roots)}')
assert len(roots) == 240   # E8 has 240 roots
assert all(r.q() == -2 for r in roots)

sv = E8m.shortest_vector()
print(f'shortest vector: q = {sv.q()}')
assert sv.q() == -2

E8: sig=(0, 8), maximum=-2
roots (vectors of square -2): 240


shortest vector: q = -2


## Task 6 — "Closest vector problem on $E_{8}$"

CVP is central to decoding, reduction theory, and computing with
periods. The spike provides exact CVP and Babai's approximate CVP on
the positive-definite kernel. The reduction/CVP suite is positive-definite
vocabulary, so we use the **positive** crystallographic $E_{8}$ —
`Lattice('E8(-1)')` in the AG convention (the $-1$ twist of the negative
default).

In [15]:
E8 = Lattice('E8(-1)', label='E8')   # positive crystallographic E8 (the -1 twist)

# Short vectors: E8 has 240 roots of square 2
roots = E8.vectors_of_square(2)
print(f'E8 roots: {len(roots)}')
assert len(roots) == 240

sv = E8.shortest_vector()
print(f'shortest vector: q = {sv.q()}, minimum = {E8.minimum()}')
assert E8.minimum() == 2

# Babai approximate CVP (fast on any rank)
target = [QQ(1)/2]*8
approx = E8.approximate_closest_vector(target)
print(f'\nBabai CVP to {target}: q = {approx.q()}')

# Geometry-of-numbers
print(f'\nVolume: {E8.volume()}')
print(f'Hadamard ratio: {E8.hadamard_ratio()}')

# Exact CVP is available but expensive on rank 8; demonstrate on A2 instead
A2 = Lattice('A2(-1)', label='A2')   # positive A2 for the positive-definite CVP suite
cvp = A2.closest_vector([QQ(2)/3, QQ(2)/3])
print(f'\nA2 exact CVP to [2/3, 2/3]: {cvp}, q = {cvp.q()}')
assert cvp.q() == 2

E8 roots: 240
shortest vector: q = 2, minimum = 2

Babai CVP to [1/2, 1/2, 1/2, 1/2, 1/2, 1/2, 1/2, 1/2]: q = 34

Volume: 1
Hadamard ratio: (1/16)^(1/8)



A2 exact CVP to [2/3, 2/3]: e_0 + e_1, q = 2


## Task 7 — "$O(L)$ and its action on the discriminant group"

$O(L)$ acts on $A_{L}$ functorially — the engine behind Nikulin's gluing
theory and the classification of distinguished subgroups.

For $U$ (unimodular), $A_{U} = 0$ so the action is trivial. For a lattice
with nontrivial discriminant like $A_{2}$ ($A_{A_{2}} \cong
\mathbb{Z}/3\mathbb{Z}$), the action is nontrivial: $O(A_{2}) \cong
$D_{6}$ (order 12) acts on $\mathbb{Z}/3$ via the determinant sign.

In [16]:
# U: unimodular, disc action trivial
U = Lattice('U', label='U')
O_U = U.isometry_group()
swap = O_U.from_matrix(matrix(ZZ, 2, 2, [0, 1, 1, 0]))
assert O_U.discriminant_action(swap).is_identity()
print('U (unimodular): O(U) acts trivially on A_U = 0 ✓')

# A2: disc group Z/3, O(A2) = S3 × C2 (order 12) acts via det sign
A2 = Lattice('A2', label='A2')
O_A2 = A2.isometry_group()
print(f'\nO(A2): finite={O_A2.is_finite()}, order={O_A2.order()}')
assert O_A2.is_finite() and O_A2.order() == 12

# The Weyl group W(A2) ≅ S3 sits inside O(A2); reflections generate it
e0 = A2.gen(0)
sigma = A2.reflection(e0)
print(f'reflection σ_{{e0}}(e0) = {sigma(e0)}  (should be -e0)')
assert sigma(e0) == -e0

# The discriminant group of A2
A_A2 = A2.discriminant_group()
print(f'\nA_{{A2}} invariants: {A_A2.invariants()}, |A| = {A_A2.cardinality()}')
g = A_A2.gen(0)
print(f'q(g) = {A_A2.q(g)}  (in QQ/2ZZ)')
print(f'b(g,g) = {A_A2.b(g, g)}  (in QQ/ZZ)')

U (unimodular): O(U) acts trivially on A_U = 0 ✓

O(A2): finite=True, order=12
reflection σ_{e0}(e0) = -e_0  (should be -e0)

A_{A2} invariants: (3,), |A| = 3
q(g) = 4/3  (in QQ/2ZZ)
b(g,g) = 1/3  (in QQ/ZZ)


## Task 8 — "Standalone discriminant forms"

Discriminant forms can be constructed standalone — not sourced from a
lattice — via `TorsionQuadraticForm` and
`SyntheticBilinearDiscriminantForm`. This is how you specify a genus
abstractly and then ask whether a lattice realizes it.

In [17]:
# Standalone quadratic discriminant form on Z/2
D = TorsionQuadraticForm(matrix.diagonal(QQ, [QQ(1)/2]))
z = D.gen(0)
print(f'Standalone TQF: invariants={D.invariants()}')
print(f'  q(z) = {D.q(z)}   (in QQ/2ZZ)')
print(f'  b(z, z) = {D.b(z, z)}   (in QQ/ZZ)')
assert (D.q(z) - D.b(z, z)) in ZZ   # q refines b

# Standalone bilinear form induces its diagonal quadratic form
B = SyntheticBilinearDiscriminantForm(matrix.diagonal(QQ, [QQ(1)/2, QQ(1)/3]))
print(f'\nBilinear form: invariants={B.invariants()}')
x = B.gen(0)
print(f'  q(x) = b(x,x) = {B.q(x)}')
print('Bilinear → quadratic induced ✓')

Standalone TQF: invariants=(2,)


  q(z) = 1/2   (in QQ/2ZZ)
  b(z, z) = 1/2   (in QQ/ZZ)

Bilinear form: invariants=(2, 3)
  q(x) = b(x,x) = 1/2
Bilinear → quadratic induced ✓


# Coda — Axiom gating

Vocabulary placement by category membership: an operation exists
exactly where the mathematics defines it. Dual needs nondegeneracy;
genus needs integral nondegenerate; the enumeration suite needs
positive-definite (or its negative-definite transport).

Absence (where an operation is mathematically undefined) → the name
does not resolve. It does NOT raise. This is the spike's central design
principle: the API *is* the mathematics.

In [18]:
A2 = Lattice('A2(-1)', label='A2')   # positive definite: carries the full reduction suite
degenerate = Lattice(matrix(QQ, [[0, 0], [0, 0]]), label='deg')
rational = Lattice(matrix(QQ, [[QQ(1)/2]]), base_ring=QQ, label='half')

# Definedness gates placement/absence
for name in ('dual', 'dual_inclusion', 'discriminant_group', 'genus',
             'same_genus', 'glue', 'maximal_overlattice', 'local_modification'):
    assert hasattr(A2, name), f'{name} should resolve for A2'

# Degenerate: dual/discriminant/genus are absent (mathematically undefined)
for name in ('dual', 'discriminant_group', 'genus'):
    assert not hasattr(degenerate, name), f'{name} should NOT resolve on a degenerate lattice'

# Rational (not integral): discriminant_group/genus absent
for name in ('discriminant_group', 'genus'):
    assert not hasattr(rational, name), f'{name} should NOT resolve on a non-integral lattice'

# Enumeration suite: positive-definite only
U = Lattice('U', label='U')
for name in ('short_vectors', 'shortest_vector', 'closest_vector',
             'voronoi_cell', 'BKZ', 'HKZ', 'gaussian_heuristic'):
    assert hasattr(A2, name), f'{name} should resolve on PD A2'
    assert not hasattr(U, name), f'{name} should NOT resolve on indefinite U'

print('Axiom gating verified across all specimens ✓')

Axiom gating verified across all specimens ✓


# Further reading

- `SYNTHETIC_LATTICE_MODEL.md` — the full specification of the based
  lattice model and the category axioms.
- `tests/` — literature fixtures and Sage reference contracts. Run via
  `just test` from this directory.
- `sage_lattice_feature_spike` — the sibling spike for new mathematics
  beyond Sage parity (gated by the gap ledger).